# Iterative Imputation
`Iterative Imputation` is an advanced missing value handling technique in which the missing values of each feature (column) are predicted using the other features — over multiple rounds (iterations).

## Example Dataset

| Age | Salary |
| --- | ------ |
| 25  | 50000  |
| 30  | ❌      |
| ❌   | 60000  |
| 40  | 80000  |

Missing values:

* Row 2 → Salary is missing
* Row 3 → Age is missing

---

## Step 1: Initial Imputation (Temporary Fill)

First, we temporarily fill missing values using the mean:

* Age mean = (25 + 30 + 40) / 3 = 31.67
* Salary mean = (50000 + 60000 + 80000) / 3 = 63333

### Updated Table:

| Age   | Salary |
| ----- | ------ |
| 25    | 50000  |
| 30    | 63333  |
| 31.67 | 60000  |
| 40    | 80000  |

This is just a starting point (not the final result)

---

## Step 2: Predict Salary (Iteration 1)

Now we treat **Salary as the target variable**
and use **Age as the input feature**

The model learns an approximate relationship:

Salary ≈ 2000 × Age

### Missing Salary (Row 2):

* Age = 30
* Predicted Salary = 60000

### Updated Table:

| Age   | Salary |
| ----- | ------ |
| 25    | 50000  |
| 30    | 60000  |
| 31.67 | 60000  |
| 40    | 80000  |

---

## Step 3: Predict Age (Iteration 1)

Now we treat **Age as the target variable**
and use **Salary as the input**

The model learns:

Age ≈ Salary / 2000

### Missing Age (Row 3):

* Salary = 60000
* Predicted Age = 30

### Updated Table:

| Age | Salary |
| --- | ------ |
| 25  | 50000  |
| 30  | 60000  |
| 30  | 60000  |
| 40  | 80000  |

---

## Step 4: Next Iterations

Now the same process is repeated:

1. Predict Salary
2. Predict Age

Since the values are no longer changing, they are considered stable
This is called convergence

---

## Final Output

| Age | Salary |
| --- | ------ |
| 25  | 50000  |
| 30  | 60000  |
| 30  | 60000  |
| 40  | 80000  |

---

## Concept Explanation

This method is based on Multivariate Imputation by Chained Equations

### Core Idea:

* Each column is treated as a target one by one
* Other columns are used to predict its missing values
* This process runs for multiple iterations

---

## Quick Workflow (Revision)

1. Identify missing values
2. Perform initial fill (mean/median)
3. Select one column as target
4. Train model and predict missing values
5. Repeat for next column
6. Run multiple iterations
7. Stop when values stabilize

---

## Easy Way to Remember

Predict each column using the others repeatedly until the values stop changing


# Let's Code !!!!!!!!!

Imports `pandas`, `numpy`, and `LinearRegression` for data manipulation and modeling.

In [201]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

Removes the 'Profit' column from the DataFrame, preparing independent variables for imputation.

In [248]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [224]:
df = df.iloc[:,0:-1]
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


Deliberately introduces `NaN` values to simulate missing data at specific locations.

In [225]:
df.iloc[1,0] = np.NaN
df.iloc[3,1] = np.NaN
df.iloc[-1,-1] = np.NaN

In [226]:
df.head()

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


Initializes `df0` by imputing all `NaN` values with their respective column means.

In [227]:
# Step 1 - Impute all missing values with mean of respective col

df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [228]:
# 0th Iteration
df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


Creates `df1` (copy of `df0`) and resets the first missing 'R&D Spend' value to `NaN`.

In [229]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.NaN

df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


Defines `X` using 'Administration' and 'Marketing Spend' to predict 'R&D Spend'.

In [230]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


Defines `y` as the 'R&D Spend' column for the regression model.

In [231]:
y = df1.iloc[[0,2,3,4],0]
y

21     8.0
2     15.0
14    12.0
44     2.0
Name: R&D Spend, dtype: float64

Trains a `LinearRegression` model and predicts the missing 'R&D Spend' value as `23.14158651`.

In [232]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

array([23.14158651])

Assigns the predicted `R&D Spend` value (`23.14`) back to `df1`.

In [233]:
df1.iloc[1,0] = 23.14

In [234]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


Resets the `Administration` value at `df1.iloc[3,1]` to `NaN` for the next imputation.

In [235]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.NaN

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.14,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,29.25


Defines `X` using 'R&D Spend' and 'Marketing Spend' to predict 'Administration'.

In [236]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[[0,1,2,4],[0,2]]
X

,R&D Spend,Marketing Spend
21,8.00,30.00
37,23.14,20.00
2,15.00,41.00
44,2.00,29.25


Defines `y` as the 'Administration' column for the regression model.

In [237]:
y = df1.iloc[[0,1,2,4],1]
y

21    15.0
37     5.0
2     10.0
44    15.0
Name: Administration, dtype: float64

Trains a `LinearRegression` model and predicts the missing 'Administration' value as `11.06331285`.

In [238]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

array([11.06331285])

Assigns the predicted 'Administration' value (`11.06`) back to `df1`.

In [239]:
df1.iloc[3,1] = 11.06

In [240]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


Resets the 'Marketing Spend' value at `df1.iloc[4,-1]` to `NaN` for the final imputation.

In [241]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.NaN

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.0
37,23.14,5.00,20.0
2,15.00,10.00,41.0
14,12.00,11.06,26.0
44,2.00,15.00,NaN


Defines `X` using 'R&D Spend' and 'Administration' to predict 'Marketing Spend'.

In [242]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06


Defines `y` as the 'Marketing Spend' column for the regression model.

In [243]:
y = df1.iloc[0:4,-1]
y

21    30.0
37    20.0
2     41.0
14    26.0
Name: Marketing Spend, dtype: float64

Trains a `LinearRegression` model and predicts the missing 'Marketing Spend' value as `31.56351448`.

In [244]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

array([31.56351448])

Assigns the predicted 'Marketing Spend' value (`31.56`) back to `df1`.

In [245]:
df1.iloc[4,-1] = 31.56

In [246]:
# After 1st Iteration
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


Calculates `df1 - df0` to show changes from the initial mean imputation.

In [247]:
# Subtract 0th iteration from 1st iteration

df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


Creates `df2` (copy of `df1`) and resets 'R&D Spend' at `df2.iloc[1,0]` to `NaN` for the second iteration. And follow the steps till our `Nan` value difference come close to the original value or it would be `0`.

In [249]:
df2 = df1.copy()

df2.iloc[1,0] = np.NaN

df2

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.06,26.00
44,2.0,15.00,31.56


In [250]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

array([23.78627207])

In [252]:
df2.iloc[1,0] = 23.78

In [253]:
df2.iloc[3,1] = np.NaN
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

array([11.22020174])

In [254]:
df2.iloc[3,1] = 11.22

In [255]:
df2.iloc[4,-1] = np.NaN

X = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

array([38.87979054])

In [256]:
df2.iloc[4,-1] = 31.56

In [257]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,31.56


In [258]:
df2 - df1

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.0
37,0.64,0.00,0.0
2,0.00,0.00,0.0
14,0.00,0.16,0.0
44,0.00,0.00,0.0


In [259]:
df3 = df2.copy()

df3.iloc[1,0] = np.NaN

df3

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.22,26.00
44,2.0,15.00,31.56


In [260]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

array([24.57698058])

In [261]:
df3.iloc[1,0] = 24.57

In [262]:
df3.iloc[3,1] = np.NaN
X = df3.iloc[[0,1,2,4],[0,2]]
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

array([11.37282844])

In [265]:
df3.iloc[3,1] = 11.37

In [266]:
df3.iloc[4,-1] = np.NaN

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

array([45.53976417])

In [272]:
df3.iloc[4,-1] = 45.53

In [270]:
df2.iloc[3,1] = 11.22

In [273]:
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,24.57,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.37,26.00
44,2.00,15.00,45.53


In [275]:
df3 - df2

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.79,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.15,0.00
44,0.00,0.00,13.97
